# Welcome to Python for fMRI analysis

This notebook is a gentle first step into Python for working with neuroimaging data. You do **not** need prior programming experience, I've made the notebook to be as beginner friendly as possible to make sure everyone can dabble in python and also gain some fluency and experience with it.

By the end, you will be able to:

- prepare a Python environment on our cluster (don't worry its very easy and its recommended);
- install and import the main scientific Python libraries;
- work with variables, lists, dictionaries, and file paths (the main python data types);
- write simple functions and inspect tabular data with pandas;
- understand how these building blocks connect to fMRI analysis.

Some setup commands below are specific to our cluster. Run them in a **terminal**, not in a Python notebook cell. The Python examples can be run in notebook cells.

## 1. Set up Python on the Linux cluster

A **module** is a centrally managed software installation. Loading a Python module makes a cluster-provided Python available in your shell terminal making sure its the clusters python, not your computers.

Open a terminal and run:

```bash
# See which Python modules are available.
module avail python

# as you can see  python/3.14-anaconda-2026.07 is the default, so if you just type below, you'll load the latest version of python
module load python

# now you might wonder, do these versions matter? They do, for instance some python libraries simply do not work on older python versions. For the most part 3.14 is very backwards compatibile and its not problematic at all for new libraries.
# but lets say, you need to use a particular library that only works with python 3.10, you can load the python module again, but this time be specific on the version (see what is available using module avail python)
# Replace with an available version, for example 3.10.7.
module load python/3.10.7

# No matter what you do, you should always confirm what you've loaded up even if you've written everything carefully (this is just my own trust issues showing but its good practice I promise)
# Confirm which Python is by typing:
which python
python --version
```


### Create and activate a virtual environment

A virtual environment keeps this project's libraries/packages separate from the system Python and from other projects. Create it once, then activate it whenever you work on a project like this, or better yet make a venv for each project you work on.
There isn't anything wrong with reusing older enviornments, but you should be very careful given libraries can have competing names for functions, thus leading to conflicts, or worse, you use the function thinking it was from the library you actually meant to use. Regardless, its a two minute ordeal, its worth it I promise.

```bash
# Now that you've loaded the clusters python module you can now make your venv the commands syntax is simple.
# python (you need to tell the shell you are using python), 
# -m (for make), 
# venv (virtual enviornment) 
# /projects/your_id/fMRI_basics/fmri_basics_venv (path to where you want the enviornment to be saved. Be sure the path exists!!!)

python -m venv /projects/your_id/fMRI_basics/fmri_basics_venv

# now we activate the venv using the command source
source /projects/your_id/fMRI_basics/fmri_basics_venv/bin/activate

# Your terminal should now show the name of the venv on the left. You can also do which python again to confirm
which python
python --version
```

To leave the environment, run `deactivate`. To use it again later, load the cluster Python module and run the `source` command again.

### Install the main libraries

With the virtual environment activated, you can now install Python packages that you want to use for your respective project, or in this case for this tutorial, which are common in typical fMRI workflows:
Important to note! these packages/libraries are going to be silo'd and will not interact with the computers system level python, so don't hesitate to install python packages, you won't break your computer

```bash

# pip is the command you use to install python packages (pip = ackage installer for Python, it connects to https://pypi.org/ the online python repository of packages)
python -m pip install --upgrade pip # this just updates pip to the latest version 
python -m pip install numpy pandas scipy matplotlib seaborn nibabel nilearn jupyter ipykernel
```

What these packages are for:

- `numpy`: numerical arrays and matrix operations;
- `pandas`: tables such as TSV confounds and parcellated time series;
- `scipy`: scientific and statistical methods;
- `matplotlib` and `seaborn`: plots;
- `nibabel`: reading and writing NIfTI neuroimaging files;
- `nilearn`: common neuroimaging analysis and visualization tools;
- `jupyter` and `ipykernel`: running this notebook with the virtual environment.

Now lets say you want to share this enviornment with someone, you can save all the installed versions to a text file with `python -m pip freeze > requirements.txt`. # this outputs the requirement.txt in whatever directory the terminal is currently in

### Make the environment available as a notebook kernel

If VS Code or Jupyter does not automatically find the virtual environment, register it once:

```bash
python -m ipykernel install --user \
    --name fmri_basics_venv \
    --display-name "Python (fmri-basics)"
```

Then select **Python (fmri-basics)** as the notebook kernel. The kernel is the Python process that executes notebook cells. The terminal environment and notebook kernel must point to the same virtual environment.

In [ ]:
import sys

import matplotlib.pyplot as plt
import nibabel as nib
import nilearn
import numpy as np
import pandas as pd
import scipy

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"SciPy: {scipy.__version__}")
print(f"nibabel: {nib.__version__}")
print(f"nilearn: {nilearn.__version__}")
print("The environment is ready for the tutorials.")

## 2. Your first Python concepts

Python executes instructions from top to bottom. A line beginning with `#` is a comment and is not executed. Run the next cell and change the values to see what happens.

In [ ]:
# Variables store values that we can use later.
subject = "sub-01" # anthing in the quotes is a string. Strings are used to store text.
number_of_runs = 2 # this data type is an integer. Integers are used to store whole numbers.
repetition_time_seconds = 2.0 # the data type is a float. Floats are used to store decimal numbers.

print(subject)
print(f"{subject} has {number_of_runs} runs")
print(f"The repetition time is {repetition_time_seconds} seconds")

### Core data structures

Lists hold an ordered collection, as in each item is numbered (indexed) and it won't change unless you make it change.
Dictionaries associate names with values. Think of it like a mapping technique e.g, if you have a column capturing sex (0 for males, 1 for females), you can make a dictionary to map 0 to the string "Male" and 1 to the string "Female"
Loops repeat an operation. 

In [ ]:
subjects = ["sub-01", "sub-02", "sub-03"] # lists are made using square brackets [ ]
acquisition = {"task": "rest", 
               "run": 1, 
               "space": "MNI152NLin2009cAsym"} # dictionaries are made using curly brackets { } and key-value pairs separated by colons.

for subject_id in subjects:
    print(f"Preparing {subject_id}")

print(acquisition["task"])
print(acquisition["space"])
print(f"There are {len(subjects)} subjects")

### Functions and file paths

Functions package reusable logic. `pathlib.Path` is the recommended way to construct paths because it works cleanly across operating systems and makes file searches readable.

As always with every package you use or see, read and look up the documentation to learn how to use it! 

https://docs.python.org/3/library/pathlib.html

In [ ]:
from pathlib import Path



def subject_func_dir(root: Path, subject_id: str) -> Path: # the colon after the function name indicates that the function is about to start. The arrow -> indicates the return type of the function.
    """Return the functional directory for one subject.""" # so this function just returns the path to the functional directory, it just takes the path to your project and the subject id as input and returns the path to the functional directory for that subject.
    return root / subject_id / "func"


project_root = Path("/projects/aabdulrasul/Tutorials")
example_func_dir = subject_func_dir(project_root / "data", "sub-01")

print(f"Project folder: {project_root}")
print(f"Example functional folder: {example_func_dir}")
print(f"Does it exist? {example_func_dir.exists()}")

## 3. Tables and arrays: the language of fMRI data

BOLD data and confounds are usually stored as tab-separated tables. In pandas, a table is a `DataFrame`; each column is a variable and each row is one time point. 

In [ ]:
# A small synthetic table: rows are time points and columns are parcels.
timepoints = np.arange(6) # this creates an array of integers from 0 to 5, representing time points in the time series data. numpy is a library for numerical computing in Python, and the arange function generates an array, which is a data structure like lists but more efficient for computing.

# this creates a pandas dataframe, which is a 2 dimensional data structure (table) that can store data of different types (strings, floats, integers)
# the dataframe is created by passing a dictionary to the pd.dataframe, the keys of the dictionary are the column names and the values are the data for each column.

example_timeseries = pd.DataFrame( 
    {
        "timepoint": timepoints, 
        "parcel_1": [0.2, 0.4, 0.1, 0.5, 0.7, 0.6],
        "parcel_2": [0.3, 0.5, 0.2, 0.4, 0.8, 0.7],
    }
)

example_timeseries

# the column on the complete left is the index, its a special column that pandas uses to identify each row, it is not part of the data itself. but rather it is a label for each row. this is important because you can always
# access the data in a dataframe by using the index, and you can also set the index to be one of the columns in the dataframe. therefore you can't lose the index column, it is always there, even if you don't see it.
# so when you're manipulating dataframes and you lose track of the row you're on, you can always use the index to get back to the row you want.

In [ ]:
# Pandas has a lot of builtin functions that can be used to manipulate dataframes, view data, and compute statistics. Here are a few examples.
print("First three rows:")
display(example_timeseries.head(3)) # the head function returns the first n rows of the dataframe, where n is the number passed to the function.
                                    # display is a function that is used to display the output of a cell in a fancy way. It is similar to the print function, but it can display more complex objects like dataframes and plots.

print(f"Shape: {example_timeseries.shape}") # shape tells us how the dataframe is structured, as in how many rows and columns it has 
print("Column names:", list(example_timeseries.columns)) # you can also list out all the column names in a dataframe too
print("\nSummary statistics:") # and you can also get summary statistics for each column in the dataframe, like the mean, standard deviation, min, max, and quartiles.
display(example_timeseries.describe())

## 4. A first plot

Python is very powerful and can not only, manipulate data and organize things, but it can also visualize.
We can plot the two synthetic parcel time series. using the matplotlib library

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4)) # fig is the figure object, which is the entire figure that contains all the axes, labels, titles, and other elements.
                                       # ax is the axes object, which is the area where the data is plotted.
                                       # plt.subplots is a function that creates a figure and a set of subplots (axes) in one call. The figsize argument specifies the size of the figure in inches (width, height).
ax.plot(example_timeseries["timepoint"], example_timeseries["parcel_1"], marker="o", label="Parcel 1") 
# this plots the data in a line using the dataframe, using the timepoint column as the x-axis and the parcel_1 column as the y-axis. The marker argument specifies that we want to use circles to mark each data point, and the label argument specifies the label for this line in the legend.
ax.plot(example_timeseries["timepoint"], example_timeseries["parcel_2"], marker="o", label="Parcel 2")
# this plots the another line of data in the dataframe, using the timepoint column as the x-axis and the parcel_2 column as the y-axis. The marker argument specifies that we want to use circles to mark each data point, and the label argument specifies the label for this line in the legend.
ax.set_xlabel("Time point")
# this sets the x axis label
ax.set_ylabel("Signal")
# this sets the y axis label
ax.set_title("Example parcel time series")
# this sets the title of the plot
ax.legend()
# this adds a legend to the plot, which shows the labels for each line in the plot
fig.tight_layout()
# this just adjusts the layout of the plot so that the labels and titles don't overlap with the plot itself
plt.show()